In [ ]:
!pip install torch torchvision torchaudio
!pip install tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
print(f"Success! Torch version: {torch.__version__}")
from torchvision import transforms, models
import pandas as pd
import numpy as np
from PIL import Image
import os
from sklearn.preprocessing import MinMaxScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
from torchvision.models import ResNet18_Weights
class MultimodalModel(nn.Module):
    def __init__(self, num_tabular_features):
        super(MultimodalModel, self).__init__()

        self.cnn = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        self.cnn.fc = nn.Identity()

        self.tab_mlp = nn.Sequential(
            nn.Linear(num_tabular_features, 16),
            nn.ReLU(),
            nn.Linear(16, 16)
        )

        self.regressor = nn.Sequential(
            nn.Linear(512 + 16, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, tab, img):
        img_features = self.cnn(img)
        tab_features = self.tab_mlp(tab)

        combined = torch.cat((img_features, tab_features), dim=1)

        return self.regressor(combined)

In [ ]:
def train_model(model, dataloader, criterion, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for tab, img, label in dataloader:
            tab, img, label = tab.to(device), img.to(device), label.to(device)

            optimizer.zero_grad()
            outputs = model(tab, img)
            loss = criterion(outputs.squeeze(), label)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(dataloader):.4f}")

In [ ]:
from tqdm import tqdm
model = MultimodalModel(num_tabular_features=len(numeric_cols)).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 1
print("Model Training")

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for tab, img, label in pbar:
        tab, img, label = tab.to(device), img.to(device), label.to(device)

        optimizer.zero_grad()
        outputs = model(tab, img)
        loss = criterion(outputs.view(-1), label.view(-1))

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} Complete. Average Loss: {train_loss/len(train_loader):.2f}")

torch.save(model.state_dict(), "house_model.pth")
print("Model saved as house_model.pth")

In [ ]:
model.eval()
val_loss = 0.0
with torch.no_grad():
    for tab, img, label in val_loader:
        tab, img, label = tab.to(device), img.to(device), label.to(device)
        outputs = model(tab, img)
        loss = criterion(outputs.squeeze(), label)
        val_loss += loss.item()

print(f"Test Average Loss: {val_loss/len(val_loader):.2f}")